# V15 – Netzwerktechnik: Python-Recap

## 🎯 Lernziele
Nach diesem Notebook kannst du …
- die wichtigsten **Exception-Mechanismen** aus V09 sicher anwenden (`try/except/else/finally`, `raise`),
- die gängigen **eingebauten Exception-Typen** (`ValueError`, `TypeError`, `FileNotFoundError`, `KeyError`) unterscheiden und gezielt abfangen,
- **eigene Funktionen** im Stil von V10 schreiben (Default-Parameter, Keyword-Argumente, mehrere Rückgabewerte als Tupel, Docstrings),
- den Sichtbarkeitsbereich von Variablen (**LEGB-Regel**) in Kurzform erklären und typische Fehler beim Mischen von lokalem und globalem Zustand erkennen.

## ⏱️ Zeitbudget
Ca. 25 Minuten als Warm-up vor der eigentlichen Theorie.

## 🧭 TL;DR
Wir wiederholen hier nur die Python-Bausteine, die du in den V15-Aufgaben direkt brauchst: robuste Fehlerbehandlung beim Einlesen von Dateien und das saubere Kapseln kleiner Hilfsfunktionen. Alles andere bleibt außen vor – das Ziel ist ein schneller, fokussierter Wiedereinstieg.

## 📦 Voraussetzungen
- V09 (Ausnahmen & Fehlerbehandlung)
- V10 (Funktionen & Gültigkeitsbereich)
- Grundlegende Python-Syntax aus V05–V08

## 1. Exceptions – warum überhaupt?

Beim Einlesen von Netzwerk- und Konfigurationsdateien geht ständig etwas schief: Dateien fehlen, Ports sind keine Zahlen, Schlüssel im JSON heißen anders als erwartet. Damit dein Programm an solchen Stellen nicht abstürzt, **fängst du Fehler gezielt ab** und reagierst darauf – genau dafür gibt es in Python den `try`-Block.

Wir unterscheiden in V15 vor allem vier Fehlertypen:

- `ValueError` – Der Wert hat den richtigen Typ, aber eine unpassende Bedeutung (z. B. `int("abc")`).
- `TypeError` – Eine Operation passt nicht zum Typ der Operanden (z. B. `"80" + 1`).
- `FileNotFoundError` – Eine Datei existiert nicht oder der Pfad ist falsch.
- `KeyError` – Ein Dictionary enthält den erwarteten Schlüssel nicht.

In [1]:
# Ein typischer Parse-Fehler beim Port-Check
port_string = "achtzig"
try:
    port = int(port_string)
    print(f"Port als Zahl: {port}")
except ValueError as fehler:
    print(f"Kein gültiger Port: {fehler}")

Kein gültiger Port: invalid literal for int() with base 10: 'achtzig'


## 2. Die vollständige Form: `try / except / else / finally`

Die beiden optionalen Zweige `else` und `finally` werden oft übersehen, sind aber gerade beim Arbeiten mit Dateien nützlich. `else` läuft nur, wenn **keine** Exception aufgetreten ist – dort gehört die eigentliche Weiterverarbeitung hin. `finally` läuft **immer**, egal ob Erfolg oder Fehler – dort gehören Aufräumarbeiten hin (Dateien schließen, Verbindungen abbauen).

In [2]:
def lies_port(text):
    try:
        port = int(text)
    except ValueError:
        print("  → konnte nicht als Zahl interpretiert werden")
        return None
    else:
        print("  → Parsing erfolgreich")
        return port
    finally:
        print("  → (finally läuft immer)")

print("Eingabe '443':")
lies_port("443")
print("\nEingabe 'abc':")
lies_port("abc")

Eingabe '443':
  → Parsing erfolgreich
  → (finally läuft immer)

Eingabe 'abc':
  → konnte nicht als Zahl interpretiert werden
  → (finally läuft immer)


## 3. `raise` – eigene Fehler auslösen

Wenn eine Funktion unerlaubten Input bekommt, darfst du (und solltest du) **selbst** eine Exception werfen. Das ist sauberer als ein Rückgabewert `None`, weil der Aufrufer den Fehler nicht übersehen kann.

In [3]:
def pruefe_port_bereich(port):
    if not (1 <= port <= 65535):
        raise ValueError(f"Port {port} liegt außerhalb 1..65535")
    return port

try:
    pruefe_port_bereich(70000)
except ValueError as e:
    print(f"Abgefangen: {e}")

Abgefangen: Port 70000 liegt außerhalb 1..65535


## 4. Dateien robust öffnen

Fehlende Dateien erkennst du an `FileNotFoundError`. Der `with`-Block sorgt dafür, dass die Datei auch bei einer Exception sauber geschlossen wird – eine zusätzliche `finally`-Klausel brauchst du dafür nicht.

In [4]:
try:
    with open("datei_die_nicht_existiert.txt", encoding="utf-8") as f:
        inhalt = f.read()
except FileNotFoundError as e:
    print(f"Datei fehlt: {e.filename}")

Datei fehlt: datei_die_nicht_existiert.txt


## 5. Funktionen – Default- und Keyword-Parameter (Recap V10)

Funktionen kapseln Wissen in wiederverwendbaren Blöcken. In V15 wirst du viele kleine Funktionen schreiben, die jeweils **eine** Aufgabe erledigen (z. B. „ist das ein gültiger Port?

: 
,
: null,
: {},
: [],
: [
/", schema="https", port=443):
    """Baut eine URL aus ihren Bestandteilen zusammen.

    Gibt die zusammengesetzte URL als String zurück.
    """
    return f"{schema}://{host}:{port}{pfad}"

print(baue_url("beispiel.de"))
print(baue_url("api.intern", pfad="/v1/status", port=8080))

## 6. Mehrere Rückgabewerte als Tupel

Python-Funktionen können mehrere Werte auf einmal zurückgeben. Technisch ist das ein **Tupel**, das beim Aufruf sofort wieder zerlegt werden kann (Tuple-Unpacking).

In [5]:
def zerlege_adresse(adresse):
    """Zerlegt 'host:port' in (host, port)."""
    host, port = adresse.split(":")
    return host, int(port)

h, p = zerlege_adresse("192.168.10.200:4840")
print(f"Host={h}, Port={p}")

Host=192.168.10.200, Port=4840


## 7. Gültigkeitsbereich: LEGB in einem Satz

Python sucht Variablennamen in dieser Reihenfolge: **L**ocal (innerhalb der aktuellen Funktion) → **E**nclosing (umschließende Funktion, falls verschachtelt) → **G**lobal (Modulebene) → **B**uilt-in (Sprach-Eingebautes wie `len`). Für V15 reicht die Faustregel: *Was du in einer Funktion mit `=` neu zuweist, ist dort lokal und lebt nur bis zum `return`.*

In [6]:
zaehler = 0  # global

def erhoehe_lokal():
    zaehler = 99  # neue LOKALE Variable, überschreibt global NICHT
    return zaehler

print("Lokal:", erhoehe_lokal())
print("Global danach:", zaehler)

Lokal: 99
Global danach: 0


## 8. Mini-Übung 1 – Port-Parser

Schreibe eine Funktion `parse_port(text)`, die einen String entgegennimmt. Wenn der String eine Ganzzahl im Bereich `1..65535` ist, gib sie als `int` zurück. Sonst wirf `ValueError`. Nutze `try/except` **nicht** in der Funktion selbst – die Exception soll nach oben durchgereicht werden.

In [7]:
def parse_port(text):
    # TODO: implementiere die Prüfung
    return 0  # Platzhalter

# ▶️ Selbstkontrolle
try:
    assert parse_port("80") == 80, "Normaler Port schlägt fehl"
    try:
        parse_port("abc")
        raise AssertionError("'abc' hätte ValueError auslösen müssen")
    except ValueError:
        pass
    print("✅ Übung 1 gelöst.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Funktion `parse_port` existiert noch nicht.")

❌ Noch nicht ganz: Normaler Port schlägt fehl


## 9. Mini-Übung 2 – Adresse zerlegen

Schreibe `zerlege(adresse)`, das aus einem String der Form `"host:port"` ein Tupel `(host, port_als_int)` macht. Wenn der Doppelpunkt fehlt, soll `ValueError` ausgelöst werden.

In [8]:
def zerlege(adresse):
    # TODO: aufteilen und Port in int umwandeln
    return ("", 0)  # Platzhalter

# ▶️ Selbstkontrolle
try:
    assert zerlege("localhost:8080") == ("localhost", 8080)
    print("✅ Übung 2 gelöst.")
except AssertionError as e:
    print(f"❌ Noch nicht ganz: {e}")
except NameError:
    print("❌ Die Funktion `zerlege` existiert noch nicht.")
except ValueError:
    print("❌ Auf dem Standardfall sollte kein ValueError kommen.")

❌ Noch nicht ganz: 


## 🧾 Cheat-Sheet V15-Python-Recap

| Thema | Kurzform | Beispiel |
|---|---|---|
| Exception abfangen | `try / except Typ as e` | `except ValueError as e:` |
| Erfolgszweig | `else` nach `except` | läuft nur ohne Fehler |
| Aufräumen | `finally` | läuft immer |
| Fehler werfen | `raise ValueError("msg")` | explizit melden |
| Datei öffnen | `with open(pfad) as f:` | schließt automatisch |
| Default-Arg | `def f(x, y=10):` | optionaler Parameter |
| Mehrere Returns | `return a, b` | Tupel + Unpacking |
| LEGB | Local → Enclosing → Global → Built-in | Namensauflösung |

## ✅ Zusammenfassung
- Exceptions sind das Mittel der Wahl, wenn Eingabedaten unzuverlässig sind.
- Kleine, gut benannte Funktionen mit klarer Signatur machen die V15-Aufgaben leichter.
- Die LEGB-Regel erklärt, warum Zuweisungen in Funktionen globale Variablen nicht automatisch ändern.

## ➡️ Nächster Schritt
Weiter mit [01_theorie.ipynb](01_theorie.ipynb).